<a href="https://colab.research.google.com/github/Cairo-Henrique/Network-Value-Investing/blob/main/Value_Investing_with_Financial_indicators_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Value Investing with Financial Indicators Tool**

This notebook provides a systematic framework for evaluating stocks based on **Value Investing** principles. It automates the collection of fundamental data and scores companies against established financial benchmarks to identify potentially undervalued assets with strong balance sheets.

---

## **Methodology**

Each ticker is analyzed using **16 financial indicators**. The tool uses a **Relative Valuation** approach where most indicators are scored based on their **Z-Score** (distance from the group average) rather than static limits:

1. **Z-Score Normalization:** Indicators are normalized against the mean and standard deviation of the selected group, highlighting companies performing above or below the peer average.
2. **Directional Logic:** The tool interprets whether a high or low value is "Good" based on financial nature (e.g., lower P/E and higher ROE are better).
3. **Graham Fair Price:** A specialized absolute rule where a stock is labeled "Good" if its current price is below the calculated Graham Fair Price.

---

## **Notebook Features**

1. **Automated Data Retrieval:** Pulls metadata (`ticker.info`) and historical daily closing prices using the Yahoo Finance API.
2. **Quantitative & Qualitative Assessment:** Labels indicators as "Good" or "Bad" based on relative Z-Scores and prints detailed breakdowns for each company.
3. **Critical Warnings:** Automatically generates alerts for high-risk conditions such as:
* Negative operating or net margins.
* Excessive debt (Debt/Equity > 150).
* Low liquidity (Current Ratio < 1).
* Negative equity (Price/Book < 0).
* Shareholder value destruction (Negative ROE) and potential "Dividend Traps".


4. **Comparative Ranking:** Generates a ranking table sorted by a **Score** (ratio of "Good" indicators) and the **Mean |Z| of Good Indicators** (strength of the positive signals).
5. **Performance Visualization:** Plots the historical normalized cumulative returns of the selected assets since 2020 using interactive `plotly` charts.

---

## **Libraries Used**

* `yfinance`: For fetching real-time financial metadata and historical price series.
* `pandas`: For data structuring, vector calculations (Z-Scores), and ranking logic.
* `plotly`: For interactive visualization of historical asset performance.
* `datetime`: For graph time period

---

### **How to Use**

1. **Setup:** Run the initial cells to import libraries and define the `THRESHOLDS` dictionary.
2. **Input:** Update the `tickers` list with the symbols you wish to compare FROM THE SAME SECTOR (e.g., `['SBSP3.SA', 'SAPR11.SA', 'CSMG3.SA']`).
3. **Analysis:** Execute `compare_companies(tickers)` to view the individual indicators, critical warnings, and the final ranking table.
4. **Visualization:** Run the plotting cells to compare the market performance of the chosen tickers over time.

## Packages

In [15]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime

## Financial indicators scoring

In [16]:
# 1. Limites redefinidos para suportar a lógica do Z-Score
# 'direction': 1  (Maior é melhor -> Z-Score > 0 é 'Bom')
# 'direction': -1 (Menor é melhor -> Z-Score < 0 é 'Bom')
THRESHOLDS = {
    'trailingPE': {'direction': -1, 'label': 'P/L (Price to Earnings)'},
    'priceToBook': {'direction': -1, 'label': 'P/VPA (Price to Book)'},
    'returnOnEquity': {'direction': 1, 'label': 'ROE (Return on Equity)'},
    'returnOnAssets': {'direction': 1, 'label': 'ROA (Return on Assets)'},
    'debtToEquity': {'direction': -1, 'label': 'Dívida/Patrimônio'},
    'currentRatio': {'direction': 1, 'label': 'Current Ratio'},
    'profitMargins': {'direction': 1, 'label': 'Margem Líquida'},
    'operatingMargins': {'direction': 1, 'label': 'Margem Operacional'},
    'dividendYield': {'direction': 1, 'label': 'Dividend Yield'},
    'enterpriseToEbitda': {'direction': -1, 'label': 'EV/EBITDA'}, # Ignora a estrutura de capital (dívida), melhor que P/L para comparar empresas do mesmo setor.
    'pegRatio': {'direction': -1, 'label': 'PEG Ratio'},           # Relaciona o P/L com o crescimento esperado. PEG < 1 costuma indicar ação descontada.
    'revenueGrowth': {'direction': 1, 'label': 'Crescimento da Receita'}, # Mostra se a empresa está conseguindo vender mais (Top-line).
    'earningsGrowth': {'direction': 1, 'label': 'Crescimento do Lucro'},  # Mostra eficiência em transformar receita em lucro (Bottom-line).
    'beta': {'direction': -1, 'label': 'Beta (Risco Sistêmico)'},         # Fundamental para otimização de risco em portfólios. Beta < 1 = menos volátil que o mercado.
    'quickRatio': {'direction': 1, 'label': 'Liquidez Seca (Quick Ratio)'}, # Mais rigoroso que o Current Ratio, pois exclui estoques da conta de liquidez.
    'grahamFairPrice': {'direction': 0, 'label': 'Preço Justo (Graham)'} # 0 indica avaliação absoluta, sem Z-Score
}

def get_financial_indicators(ticker_symbol):
    """
    Retorna um dicionário com indicadores financeiros de uma empresa, dado seu ticker.
    """
    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info

    eps = info.get('trailingEps')
    growth = info.get('earningsGrowth')
    graham_price = None
    if eps is not None and growth is not None:
        graham_price = eps * (8.5 + 2 * (growth * 100))

    return {
        'Ticker': ticker_symbol.upper(),
        'Nome': info.get('longName'),
        'Setor': info.get('sector'),
        'Indústria': info.get('industry'),
        'Preço Atual': info.get('currentPrice'),
        'Market Cap': info.get('marketCap'),
        # --- Indicadores ---
        'trailingPE': info.get('trailingPE'),
        'priceToBook': info.get('priceToBook'),
        'returnOnEquity': info.get('returnOnEquity'),
        'returnOnAssets': info.get('returnOnAssets'),
        'debtToEquity': info.get('debtToEquity'),
        'currentRatio': info.get('currentRatio'),
        'profitMargins': info.get('profitMargins'),
        'operatingMargins': info.get('operatingMargins'),
        'dividendYield': info.get('dividendYield'),
        'grahamFairPrice': graham_price,
        'enterpriseToEbitda': info.get('enterpriseToEbitda'),
        'pegRatio': info.get('pegRatio'),
        'revenueGrowth': info.get('revenueGrowth'),
        'earningsGrowth': info.get('earningsGrowth'),
        'beta': info.get('beta'),
        'quickRatio': info.get('quickRatio'),
    }

def evaluate_indicators(indicators, z_scores):
    """
    Avalia cada indicador com base na direção do Z-Score.
    Retorna um dicionário com status: 'Bom', 'Ruim' ou 'Dados não disponíveis'.
    """
    avaliacoes = {}
    for key, params in THRESHOLDS.items():
        value = indicators.get(key)
        direction = params['direction']
        label = params['label']

        if value is None:
            status = 'Dados não disponíveis'
        elif key == 'grahamFairPrice':
            # Preço de Graham continua sendo uma regra absoluta de limite
            preco_atual = indicators.get('Preço Atual')
            if preco_atual is None or value is None:
                status = 'Dados não disponíveis'
            else:
                status = 'Bom' if preco_atual < value else 'Ruim'
        else:
            z_val = z_scores.get(key)
            if pd.isna(z_val):
                status = 'Dados não disponíveis'
            else:
                # Se a direção do indicador e o sinal do Z-score combinarem, o ativo está operando de forma positiva
                status = 'Bom' if (z_val * direction) > 0 else 'Ruim'

        avaliacoes[label] = status
    return avaliacoes

def compare_companies(tickers):
    """
    Compara várias empresas, calcula Z-scores para normalização,
    mostra indicadores e retorna ranking.
    """
    resultados = []
    dados_coletados = []

    # ETAPA 1: Coletar os dados de todas as empresas primeiro
    for t in tickers:
        try:
            data = get_financial_indicators(t)
            dados_coletados.append(data)
        except Exception:
            print(f"Erro ao obter dados para {t.upper()}.")
            dados_coletados.append({'Ticker': t.upper(), 'Nome': None, 'Erro': True})

    # ETAPA 2: Calcular Z-Scores usando Pandas para as empresas válidas
    df_bruto = pd.DataFrame([d for d in dados_coletados if 'Erro' not in d])
    z_scores_df = pd.DataFrame(index=df_bruto.index)

    for key in THRESHOLDS.keys():
        if key != 'grahamFairPrice' and key in df_bruto.columns:
            desvio = df_bruto[key].std()
            # Valida se o desvio não é nulo ou zero (ex: quando todas empresas tem o mesmo valor)
            if pd.notna(desvio) and desvio > 0:
                z_scores_df[key] = (df_bruto[key] - df_bruto[key].mean()) / desvio
            else:
                z_scores_df[key] = 0.0

    # ETAPA 3: Avaliar cada empresa com base no Z-Score e exibir resultados
    for i, row in df_bruto.iterrows():
        data = row.to_dict()
        z_scores_dict = z_scores_df.iloc[i].to_dict() if not z_scores_df.empty else {}

        avals = evaluate_indicators(data, z_scores_dict)

        print(f"\n=== Indicadores de {data['Ticker']} ({data['Nome']}) ===")
        for key, val in THRESHOLDS.items():
            label = val['label']
            valor = data.get(key)
            status = avals[label]

            # Formatação para mostrar o valor do Z-Score ao lado (apenas para contexto)
            z_info = f" (Z-Score: {z_scores_dict.get(key, 0):.2f})" if key != 'grahamFairPrice' and pd.notna(z_scores_dict.get(key)) else ""
            print(f"{label}: {valor}{z_info} -> {status}")

        # --- AVISOS ---
        avisos = []
        if data.get('operatingMargins') is not None and data.get('operatingMargins') < 0:
            avisos.append("⚠️ ALERTA: Margem Operacional Negativa")
        if data.get('profitMargins') is not None and data.get('profitMargins') < 0:
            avisos.append("⚠️ ALERTA: Lucro Líquido Negativo (Margem Líquida < 0)")
        if data.get('debtToEquity') is not None and data.get('debtToEquity') > 150:
            avisos.append(f"⚠️ ALERTA: Dívida Excessiva (Dívida/Patrimônio = {data.get('debtToEquity')})")
        if data.get('currentRatio') is not None and data.get('currentRatio') < 1:
            avisos.append(f"⚠️ ALERTA: Liquidez Muito Baixa (Current Ratio = {data.get('currentRatio')})")
        if data.get('priceToBook') is not None and data.get('priceToBook') < 0:
            avisos.append("⚠️ ALERTA: Patrimônio Líquido Negativo (P/VPA < 0)")
        if data.get('returnOnEquity') is not None and data.get('returnOnEquity') < 0:
            avisos.append("🚨 PERIGO: Destruição de Valor ao Acionista (ROE Negativo)")
        if data.get('revenueGrowth') is not None and data.get('revenueGrowth') < 0:
            avisos.append(f"⚠️ ALERTA: Receita em Queda (Crescimento = {data.get('revenueGrowth'):.2%})")
        if data.get('earningsGrowth') is not None and data.get('earningsGrowth') < 0:
            avisos.append(f"⚠️ ALERTA: Lucros em Queda (Crescimento = {data.get('earningsGrowth'):.2%})")
        if data.get('trailingPE') is not None and data.get('trailingPE') > 100:
            avisos.append(f"⚠️ ALERTA: Valuation Esticado / Risco de Bolha (P/L = {data.get('trailingPE'):.2f})")
        if data.get('dividendYield') is not None and data.get('dividendYield') > 15: # Acima de 15%
            avisos.append(f"🚨 PERIGO: Possível 'Dividend Trap' (Yield irreal de {data.get('dividendYield'):.2%})")
        if data.get('beta') is not None and data.get('beta') > 2:
            avisos.append(f"⚠️ ALERTA: Alta Volatilidade de Mercado (Beta = {data.get('beta')})")

        if avisos:
            print("--- AVISOS CRÍTICOS ---")
            for aviso in avisos:
                print(aviso)

        qtd_bom = sum(1 for s in avals.values() if s == 'Bom')
        total = len(avals.values())
        score = qtd_bom / total if total > 0 else 0

        z_bons_abs = []
        z_ruins_abs = []

        for key, z_val in z_scores_dict.items():
            if pd.notna(z_val) and key in THRESHOLDS and key != 'grahamFairPrice':
                direction = THRESHOLDS[key]['direction']
                # Verifica se o status foi "Bom" com base na direção do indicador
                if (z_val * direction) > 0:
                    z_bons_abs.append(abs(z_val))
                else:
                    z_ruins_abs.append(abs(z_val))

        # Cálculos de média e máximos dos módulos
        z_mean_bons = sum(z_bons_abs) / len(z_bons_abs) if z_bons_abs else 0
        z_max_bons = max(z_bons_abs) if z_bons_abs else 0
        z_max_ruins = max(z_ruins_abs) if z_ruins_abs else 0

        resultados.append({
            'Ticker': data['Ticker'],
            'Nome': data['Nome'],
            'Bom': qtd_bom,
            'Total Avaliados': total,
            'Score': round(score, 2),
            'Média |Z| Bons': round(z_mean_bons, 2),
            'Máx |Z| Bons': round(z_max_bons, 2),
            'Máx |Z| Ruins': round(z_max_ruins, 2)
        })

    # Adiciona na tabela as empresas que geraram exceção
    for d in dados_coletados:
        if 'Erro' in d:
            resultados.append({
                'Ticker': d['Ticker'], 'Nome': None, 'Bom': 0, 'Total Avaliados': 0, 'Score': 0,
                'Média |Z| Bons': 0, 'Máx |Z| Bons': 0, 'Máx |Z| Ruins': 0 # Ajustado
            })

    print("\n=== Ranking de Empresas ===")
    df = pd.DataFrame(resultados)
    df_ranked = df.sort_values(by=['Score', 'Média |Z| Bons'], ascending=False).reset_index(drop=True)
    return df_ranked

In [17]:
# =========================================================
# Seleção dos Ativos
# =========================================================
tickers = [
    'SBSP3.SA', 'SAPR11.SA', 'CSMG3.SA' # Saneamento
]

In [18]:
compare_companies(tickers)


=== Indicadores de SBSP3.SA (Companhia de Saneamento Básico do Estado de São Paulo - SABESP) ===
P/L (Price to Earnings): 12.603238 (Z-Score: -0.62) -> Bom
P/VPA (Price to Book): 0.5149114 (Z-Score: -0.82) -> Bom
ROE (Return on Equity): 0.21334 (Z-Score: 1.13) -> Bom
ROA (Return on Assets): 0.084989995 (Z-Score: 0.83) -> Bom
Dívida/Patrimônio: 94.673 (Z-Score: 0.71) -> Ruim
Current Ratio: 1.121 (Z-Score: -0.85) -> Ruim
Margem Líquida: 0.22215 (Z-Score: -0.08) -> Ruim
Margem Operacional: 0.34567 (Z-Score: 0.89) -> Bom
Dividend Yield: 0.53 (Z-Score: -1.08) -> Ruim
EV/EBITDA: 9.549 (Z-Score: 0.58) -> Ruim
PEG Ratio: 0.49 (Z-Score: -0.71) -> Bom
Crescimento da Receita: 0.439 (Z-Score: 1.15) -> Bom
Crescimento do Lucro: 0.872 (Z-Score: 1.08) -> Bom
Beta (Risco Sistêmico): 0.191 (Z-Score: -0.64) -> Bom
Liquidez Seca (Quick Ratio): 1.092 (Z-Score: -1.00) -> Ruim
Preço Justo (Graham): 451.76300000000003 -> Bom

=== Indicadores de SAPR11.SA (Companhia de Saneamento do Paraná - SANEPAR) ===
P/L

,Ticker,Nome,Bom,Total Avaliados,Score,Média |Z| Bons,Máx |Z| Bons,Máx |Z| Ruins
0,SBSP3.SA,Companhia de Saneamento Básico do Estado de Sã...,10,16,0.62,0.87,1.15,1.08
1,SAPR11.SA,Companhia de Saneamento do Paraná - SANEPAR,8,16,0.50,0.65,1.15,1.15
2,CSMG3.SA,Companhia de Saneamento de Minas Gerais,6,16,0.38,0.62,1.10,1.15


## **Piotroski F-Score**

O **Piotroski F-Score** é um sistema de pontuação que varia de **0 a 9** para avaliar a saúde financeira e a eficiência de uma empresa. Ao contrário de múltiplos de preço (como P/L), ele foca na **evolução interna** da companhia (Ano Atual vs. Ano Anterior).

### **Critérios de Pontuação (1 ponto cada):**

**1. Rentabilidade**

* **F1:** Lucro Líquido Positivo no ano atual.
* **F2:** Fluxo de Caixa Operacional (FCO) positivo no ano atual.
* **F3:** ROA (Retorno sobre Ativos) maior que o do ano anterior.
* **F4:** Qualidade do Lucro (FCO > Lucro Líquido).

**2. Alavancagem e Liquidez**

* **F5:** Redução da dívida de longo prazo em relação aos ativos.
* **F6:** Aumento da Liquidez Corrente (Capacidade de pagar dívidas de curto prazo).
* **F7:** Ausência de emissão de novas ações (Não diluição do acionista).

**3. Eficiência Operacional**

* **F8:** Aumento da Margem Bruta em relação ao ano anterior.
* **F9:** Aumento do Giro do Ativo (Eficiência em gerar receita com o que possui).

### **Interpretação:**

* **7 - 9:** Fundamentos **Fortes**. Melhoria clara em quase todas as áreas.
* **4 - 8:** Fundamentos **Estáveis**. Situação financeira equilibrada.
* **0 - 3:** Fundamentos **Fracos**. Sinais de deterioração operacional e risco financeiro.

---

In [25]:
def get_piotroski_score(ticker_symbol):
    """
    Calcula o Piotroski F-Score (0 a 9) para uma empresa avaliando:
    Rentabilidade, Alavancagem/Liquidez e Eficiência Operacional.
    """
    ticker = yf.Ticker(ticker_symbol)

    # Extrai os dados anuais mais recentes
    bs = ticker.balance_sheet
    inc = ticker.financials
    cf = ticker.cashflow

    # Validação de segurança: precisamos de pelo menos 2 anos de dados para comparar
    if bs.empty or inc.empty or cf.empty or len(bs.columns) < 2 or len(inc.columns) < 2 or len(cf.columns) < 2:
        return {'Ticker': ticker_symbol.upper(), 'Piotroski_Score': None, 'Erro': 'Dados históricos insuficientes'}

    # Função auxiliar para extrair dados em segurança, testando possíveis nomenclaturas
    def safe_get(df, possible_keys, col_index, default=0.0):
        for key in possible_keys:
            if key in df.index:
                val = df.loc[key].iloc[col_index]
                if pd.notna(val):
                    return float(val)
        return default

    # Índices de coluna: 0 é o ano actual (Current Year - cy), 1 é o ano anterior (Previous Year - py)

    # --- 1. DADOS DE RENTABILIDADE ---
    ni_cy = safe_get(inc, ['Net Income'], 0)
    ni_py = safe_get(inc, ['Net Income'], 1)
    ocf_cy = safe_get(cf, ['Operating Cash Flow', 'Cash Flow From Continuing Operating Activities', 'Total Cash From Operating Activities'], 0)
    ta_cy = safe_get(bs, ['Total Assets'], 0, default=1.0) # default 1.0 para evitar divisão por zero
    ta_py = safe_get(bs, ['Total Assets'], 1, default=1.0)

    # --- 2. DADOS DE ALAVANCAGEM, LIQUIDEZ E FONTE DE RECURSOS ---
    ltd_cy = safe_get(bs, ['Long Term Debt', 'Total Long Term Debt'], 0)
    ltd_py = safe_get(bs, ['Long Term Debt', 'Total Long Term Debt'], 1)

    ca_cy = safe_get(bs, ['Current Assets', 'Total Current Assets'], 0)
    ca_py = safe_get(bs, ['Current Assets', 'Total Current Assets'], 1)
    cl_cy = safe_get(bs, ['Current Liabilities', 'Total Current Liabilities'], 0, default=1.0)
    cl_py = safe_get(bs, ['Current Liabilities', 'Total Current Liabilities'], 1, default=1.0)

    shares_cy = safe_get(bs, ['Ordinary Shares Number', 'Share Issued', 'Basic Average Shares'], 0)
    shares_py = safe_get(bs, ['Ordinary Shares Number', 'Share Issued', 'Basic Average Shares'], 1)

    # --- 3. DADOS DE EFICIÊNCIA OPERACIONAL ---
    gp_cy = safe_get(inc, ['Gross Profit'], 0)
    gp_py = safe_get(inc, ['Gross Profit'], 1)
    rev_cy = safe_get(inc, ['Total Revenue', 'Operating Revenue'], 0, default=1.0)
    rev_py = safe_get(inc, ['Total Revenue', 'Operating Revenue'], 1, default=1.0)

    # ==========================================
    # CÁLCULO DOS 9 PONTOS DE PIOTROSZKI
    # ==========================================
    score = 0
    detalhes = {}

    # F1: Lucro Líquido Positivo
    f1 = ni_cy > 0
    score += int(f1)
    detalhes['F1_Lucro_Liquido_Positivo'] = f1

    # F2: Fluxo de Caixa Operacional Positivo
    f2 = ocf_cy > 0
    score += int(f2)
    detalhes['F2_Caixa_Operacional_Positivo'] = f2

    # F3: ROA actual > ROA ano anterior
    roa_cy = ni_cy / ta_cy
    roa_py = ni_py / ta_py
    f3 = roa_cy > roa_py
    score += int(f3)
    detalhes['F3_ROA_Crescente'] = f3

    # F4: Qualidade do Lucro (Fluxo de Caixa Operacional > Lucro Líquido)
    f4 = ocf_cy > ni_cy
    score += int(f4)
    detalhes['F4_Qualidade_Lucro_Caixa'] = f4

    # F5: Redução da Alavancagem (Dívida Longo Prazo / Ativos diminuiu)
    debt_ratio_cy = ltd_cy / ta_cy
    debt_ratio_py = ltd_py / ta_py
    # Se a empresa não tem dívidas em ambos os anos, pontua positivo
    f5 = debt_ratio_cy < debt_ratio_py or (debt_ratio_cy == 0 and debt_ratio_py == 0)
    score += int(f5)
    detalhes['F5_Alavancagem_Reduzida'] = f5

    # F6: Liquidez Corrente aumentou (Activo Circulante / Passivo Circulante)
    # Bancos normalmente não reportam estes campos, resultando em F6 = False
    cr_cy = ca_cy / cl_cy
    cr_py = ca_py / cl_py
    f6 = cr_cy > cr_py
    score += int(f6)
    detalhes['F6_Liquidez_Crescente'] = f6

    # F7: Ausência de Diluição (Não emitiu novas ações)
    # Permite um crescimento marginal de 1% para pacotes de stock options de diretores
    if shares_cy == 0 and shares_py == 0:
        f7 = False
    else:
        f7 = shares_cy <= (shares_py * 1.01)
    score += int(f7)
    detalhes['F7_Sem_Diluicao_Acoes'] = f7

    # F8: Margem Bruta aumentou
    gm_cy = gp_cy / rev_cy
    gm_py = gp_py / rev_py
    f8 = gm_cy > gm_py
    score += int(f8)
    detalhes['F8_Margem_Bruta_Crescente'] = f8

    # F9: Giro do Activo aumentou (Receita / Ativos)
    at_cy = rev_cy / ta_cy
    at_py = rev_py / ta_py
    f9 = at_cy > at_py
    score += int(f9)
    detalhes['F9_Giro_Activo_Crescente'] = f9

    return {
        'Ticker': ticker_symbol.upper(),
        'Piotroski_Score': score,
        'Detalhes': detalhes
    }

In [27]:
for t in tickers:
    resultado = get_piotroski_score(t)
    print(f"\n[{resultado['Ticker']}] Piotroski F-Score: {resultado['Piotroski_Score']} / 9")
    if resultado.get('Detalhes'):
        for criterio, passou in resultado['Detalhes'].items():
            print(f" - {criterio}: {'✅ Passou' if passou else '❌ Falhou'}")


[CSMG3.SA] Piotroski F-Score: 4 / 9
 - F1_Lucro_Liquido_Positivo: ✅ Passou
 - F2_Caixa_Operacional_Positivo: ✅ Passou
 - F3_ROA_Crescente: ❌ Falhou
 - F4_Qualidade_Lucro_Caixa: ✅ Passou
 - F5_Alavancagem_Reduzida: ❌ Falhou
 - F6_Liquidez_Crescente: ❌ Falhou
 - F7_Sem_Diluicao_Acoes: ✅ Passou
 - F8_Margem_Bruta_Crescente: ❌ Falhou
 - F9_Giro_Activo_Crescente: ❌ Falhou

[SAPR11.SA] Piotroski F-Score: 6 / 9
 - F1_Lucro_Liquido_Positivo: ✅ Passou
 - F2_Caixa_Operacional_Positivo: ✅ Passou
 - F3_ROA_Crescente: ✅ Passou
 - F4_Qualidade_Lucro_Caixa: ✅ Passou
 - F5_Alavancagem_Reduzida: ✅ Passou
 - F6_Liquidez_Crescente: ❌ Falhou
 - F7_Sem_Diluicao_Acoes: ✅ Passou
 - F8_Margem_Bruta_Crescente: ❌ Falhou
 - F9_Giro_Activo_Crescente: ❌ Falhou

[SBSP3.SA] Piotroski F-Score: 4 / 9
 - F1_Lucro_Liquido_Positivo: ✅ Passou
 - F2_Caixa_Operacional_Positivo: ✅ Passou
 - F3_ROA_Crescente: ❌ Falhou
 - F4_Qualidade_Lucro_Caixa: ❌ Falhou
 - F5_Alavancagem_Reduzida: ❌ Falhou
 - F6_Liquidez_Crescente: ✅ Passo

## Plot graph

In [19]:
# =========================================================
# Seleção dos Ativos
# =========================================================
# tickers =

# =========================================================
# Período de análise
# =========================================================
start_time = "2020-01-01"
end_time = datetime.now().strftime('%Y-%m-%d')

In [20]:
# Baixar dados
data = yf.download(
    tickers,
    start=start_time,
    end=end_time
)["Close"]

# =========================================================
# Limpeza & Normalização
# =========================================================
data = data.dropna(axis=1)
tickers = data.columns.tolist()
data_normalized = data / data.iloc[0]

/tmp/ipykernel_29023/2709044113.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  3 of 3 completed


In [21]:
fig = go.Figure()

# Adicionar cada ativo
for ativo in data_normalized.columns:
    fig.add_trace(go.Scatter(
        x=data_normalized.index,
        y=data_normalized[ativo],
        mode='lines',
        name=ativo
    ))

# Layout
fig.update_layout(
    title="Desempenho das Ações",
    xaxis_title="Data",
    yaxis_title="Preço Normalizado",
    xaxis=dict(rangeslider=dict(visible=True)),
    hovermode="x unified",
    template="plotly_dark",
    autosize=True,
    height=500,
)

fig.show()

## Valuation via Discounted Cash Flux

In [28]:
# =========================================================
# Função de Cálculo por Ativo
# =========================================================
def calcular_valuation_ativo(ticker, metodo_fluxo="FCF", taxa_desconto=0.10,
                             crescimento_projetado=0.08, crescimento_perpetuo=0.025,
                             anos_projecao=5):
    """
    Calcula o valor justo e a margem de segurança de um ativo via DCF.
    """
    try:
        acao = yf.Ticker(ticker)
        info = acao.info
        cf = acao.cash_flow

        if cf.empty:
            return {"Ticker": ticker, "Status": "Erro: Sem dados de fluxo de caixa"}

        # 1. Definir o fluxo base
        if metodo_fluxo == "FCF":
            if "Free Cash Flow" in cf.index:
                fluxo_base = cf.loc["Free Cash Flow"].iloc[0]
            elif "Operating Cash Flow" in cf.index and "Capital Expenditure" in cf.index:
                capex = cf.loc["Capital Expenditure"].iloc[0]
                # Garante que o CapEx seja deduzido, independentemente do sinal que a API mandar
                fluxo_base = cf.loc["Operating Cash Flow"].iloc[0] - abs(capex)
            else:
                return {"Ticker": ticker, "Status": "Erro: Dados para FCF indisponíveis"}
        elif metodo_fluxo == "OCF":
            if "Operating Cash Flow" in cf.index:
                fluxo_base = cf.loc["Operating Cash Flow"].iloc[0]
            else:
                return {"Ticker": ticker, "Status": "Erro: Dados para OCF indisponíveis"}
        else:
            raise ValueError("Método inválido. Escolha 'FCF' ou 'OCF'.")

        if pd.isna(fluxo_base) or fluxo_base <= 0:
            return {"Ticker": ticker, "Status": "Ignorado: Fluxo base <= 0"}

        # 2. Projetar fluxos de caixa
        fluxos_projetados = []
        fluxo_atual = fluxo_base
        for ano in range(1, anos_projecao + 1):
            fluxo_atual *= (1 + crescimento_projetado)
            fluxos_projetados.append(fluxo_atual)

        # 3. Trazer a valor presente (VP)
        vp_fluxos = sum([f / ((1 + taxa_desconto) ** i) for i, f in enumerate(fluxos_projetados, 1)])

        # 4. Calcular Valor Terminal e seu VP
        fluxo_terminal = fluxos_projetados[-1] * (1 + crescimento_perpetuo)
        valor_terminal = fluxo_terminal / (taxa_desconto - crescimento_perpetuo)
        vp_valor_terminal = valor_terminal / ((1 + taxa_desconto) ** anos_projecao)

        # 5. Calcular Enterprise Value e Equity Value
        enterprise_value = vp_fluxos + vp_valor_terminal

        caixa = info.get('totalCash', 0)
        divida = info.get('totalDebt', 0)

        if metodo_fluxo == "FCF":
            equity_value = enterprise_value + caixa - divida
        else:
            equity_value = enterprise_value

        # 6. Calcular Valor Justo por Ação e Margem de Segurança
        acoes_em_circulacao = info.get('sharesOutstanding')
        if not acoes_em_circulacao:
            return {"Ticker": ticker, "Status": "Erro: Ações em circulação não encontradas"}

        valor_justo = equity_value / acoes_em_circulacao
        preco_atual = info.get('currentPrice', info.get('previousClose', 0))

        if preco_atual == 0:
             return {"Ticker": ticker, "Status": "Erro: Preço atual não encontrado"}

        margem_seguranca = ((valor_justo - preco_atual) / valor_justo) * 100

        # Retorna o dicionário de sucesso
        return {
            "Ticker": ticker,
            "Preço Atual (US$)": round(preco_atual, 2),
            "Valor Justo (US$)": round(valor_justo, 2),
            "Margem de Segurança (%)": round(margem_seguranca, 2),
            "Status": "Sucesso"
        }

    except Exception as e:
        return {"Ticker": ticker, "Status": f"Erro inesperado: {e}"}


# =========================================================
# Função Gerenciadora (Processa Múltiplos Ativos)
# =========================================================
def gerar_relatorio_valuation(lista_tickers, **kwargs):
    """
    Processa uma lista de ativos e retorna um DataFrame consolidado.
    Repassa qualquer premissa (kwargs) para a função de cálculo.
    """
    print(f"Iniciando análise para {len(lista_tickers)} ativos...\n")
    resultados = []

    for ticker in lista_tickers:
        resultado_ativo = calcular_valuation_ativo(ticker, **kwargs)
        resultados.append(resultado_ativo)
        print(f"[{ticker}] Processado. Status: {resultado_ativo['Status']}")

    df = pd.DataFrame(resultados)

    # Organiza a exibição: separa os que deram sucesso dos que deram erro
    df_sucesso = df[df["Status"] == "Sucesso"].drop(columns=["Status"])
    df_erros = df[df["Status"] != "Sucesso"][["Ticker", "Status"]]

    return df_sucesso, df_erros

In [29]:
# =========================================================
# Seleção dos Ativos
# =========================================================
# tickers =

In [31]:
# Definindo as premissas do usuário
minhas_premissas = {
    "metodo_fluxo": "OCF",
    "taxa_desconto": 0.12,         # Exigindo 12% de retorno (maior rigor)
    "crescimento_projetado": 0.09, # Assumindo 9% de crescimento nos primeiros anos
    "crescimento_perpetuo": 0.03,  # 3% na perpetuidade
    "anos_projecao": 5
}

# Executa a função
df_valuation, df_falhas = gerar_relatorio_valuation(tickers, **minhas_premissas)

# Exibe os resultados
print("\n" + "="*50)
print("VALUATION CONCLUÍDO")
print("="*50)
display(df_valuation) # No Colab, 'display' renderiza uma tabela bonita

if not df_falhas.empty:
    print("\n" + "="*50)
    print("ATIVOS IGNORADOS OU COM ERRO")
    print("="*50)
    display(df_falhas)

Iniciando análise para 3 ativos...

[CSMG3.SA] Processado. Status: Sucesso
[SAPR11.SA] Processado. Status: Sucesso
[SBSP3.SA] Processado. Status: Sucesso

VALUATION CONCLUÍDO


,Ticker,Preço Atual (US$),Valor Justo (US$),Margem de Segurança (%)
0,CSMG3.SA,54.44,89.80,39.38
1,SAPR11.SA,42.33,341.15,87.59
2,SBSP3.SA,31.13,34.82,10.60
